<a href="https://colab.research.google.com/github/gauravjain6633/beam-yaml-codelab-training/blob/main/apache_beam_yaml_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Try Apache Beam - YAML
While Beam provides powerful APIs for authoring sophisticated data processing pipelines, it still has a high barrier for getting started and authoring simple pipelines. Even setting up the environment, installing the dependencies, and setting up the project can be a challenge.

Here we provide a simple declarative syntax for describing pipelines that does not require coding experience or learning how to use an SDK—any text editor will do. Some installation may be required to actually execute a pipeline, but we envision various services (such as Dataflow) to accept yaml pipelines directly obviating the need for even that in the future. We also anticipate the ability to generate code directly from these higher-level yaml descriptions, should one want to graduate to a full Beam SDK (and possibly the other direction as well as far as possible).


In this notebook, you set up your development environment and write a simple pipeline using YAML. Then you run it locally, using the DirectRunner. You can explore other runners with the Beam Capability Matrix.

To navigate through different sections, use the table of contents. From View drop-down list, select Table of contents.

To run a code cell, click the Run cell button at the top left of the cell, or select it and press Shift+Enter. Try modifying a code cell and re-running it to see what happens.

To learn more about Colab, see Welcome to Colaboratory!.

Setup
First, you need to set up your environment. The following code installs apache-beam and creates directories for your data, pipelines and results.

In [1]:
# Install apache-beam
%pip install --quiet apache-beam[yaml,tfrecord]

# Create directories for storing data, pipelines and results
! mkdir -p data
! mkdir -p pipelines
! mkdir -p results

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 816.4/816.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
%%writefile 'data/products.csv'
id,firstname,age,country,profession,category,product_name
1,Reeba,58,Belgium,unemployed,Books,The Great Novel
2,Maud,45,Spain,firefighter,Electronics,Laptop X
3,Meg,11,France,unemployed,Books,Kids Story Book
4,Rani,53,Spain,doctor,Electronics,Smartphone Y
5,Natka,26,France,doctor,Apparel,T-shirt
6,Aurore,32,Italy,police officer,Electronics,Laptop X
7,Elvira,39,Italy,doctor,Home,Chair
8,Asia,10,Belgium,doctor,Electronics,Gaming Mouse
9,Lesly,35,Spain,firefighter,Books,Science Fiction
10,Orelia,31,Germany,police officer,Electronics,Smartphone Y
11,Theodora,16,Italy,unemployed,Apparel,Jeans
12,Viviene,44,Germany,police officer,Electronics,Laptop X
13,Teriann,50,Belgium,police officer,Electronics,Gaming Mouse
14,Carol-Jean,23,Germany,unemployed,Home,Table
15,Drucie,15,Spain,police officer,Books,Fantasy Novel
16,Elie,10,Italy,doctor,Apparel,Jacket
17,Shaylyn,34,Belgium,worker,Electronics,Smartphone Y
18,Fayre,33,Spain,police officer,Books,History Book
19,Sabina,52,Germany,police officer,Electronics,Laptop Z
20,Aryn,20,Belgium,police officer,Apparel,Sweater

Writing data/products.csv


The `data/products.csv` file has been created. It contains sample product data with fields like `id`, `firstname`, `age`, `country`, `profession`, `category`, and `product_name`. This file will serve as the input for our Beam pipelines.

In [3]:
! head data/products.csv

id,firstname,age,country,profession,category,product_name
1,Reeba,58,Belgium,unemployed,Books,The Great Novel
2,Maud,45,Spain,firefighter,Electronics,Laptop X
3,Meg,11,France,unemployed,Books,Kids Story Book
4,Rani,53,Spain,doctor,Electronics,Smartphone Y
5,Natka,26,France,doctor,Apparel,T-shirt
6,Aurore,32,Italy,police officer,Electronics,Laptop X
7,Elvira,39,Italy,doctor,Home,Chair
8,Asia,10,Belgium,doctor,Electronics,Gaming Mouse
9,Lesly,35,Spain,firefighter,Books,Science Fiction


This pipeline, `pipeline-01.yaml`, is a basic example demonstrating how to read data from a CSV file and log each record. It uses two transforms:

*   **`ReadFromCsv`**: Reads the data from `data/products.csv`. Each row of the CSV will be treated as a record.
*   **`LogForTesting`**: This transform is specifically for development and debugging. It prints each element (record) of the pipeline's PCollection to the console's `INFO:root` log. You'll see each row from the CSV file printed as a JSON-like dictionary.

In [4]:
%%writefile 'pipelines/pipeline-01.yaml'
pipeline:
  type: chain
  transforms:
    - type: ReadFromCsv
      config:
        path: data/products.csv
    - type: LogForTesting

Writing pipelines/pipeline-01.yaml


We can verify the contents of this file by running:

In [5]:
! cat pipelines/pipeline-01.yaml

pipeline:
  type: chain
  transforms:
    - type: ReadFromCsv
      config:
        path: data/products.csv
    - type: LogForTesting


Now, we can execute the yaml pipeline by passing this file as an argument to the following command:

In [6]:
! python -m apache_beam.yaml.main --pipeline_spec_file=pipelines/pipeline-01.yaml

INFO:root:Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
Building pipeline...
INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.
2026-04-08 06:18:01.046587: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 06:18:01.051956: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 06:18:01.066000: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775629081.091661    4806 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775629081.099935    4806 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attemptin

Upon executing this command, the Apache Beam DirectRunner will process `pipeline-01.yaml`. You will see detailed logs in the output. Look for lines starting with `INFO:root:` followed by JSON objects, which represent each record from the `products.csv` file. This confirms that the data is being read and processed correctly by the pipeline.


If you scroll through the output logs, you'll find entries such as:

INFO:root:{"id": 1, "firstname": "Reeba", "age": 58, "country": "Belgium", "profession": "unemployed", "category": "Books", "product_name": "The Great Novel"}
INFO:root:{"id": 2, "firstname": "Maud", "age": 45, "country": "Spain", "profession": "firefighter", "category": "Electronics", "product_name": "Laptop X"}
INFO:root:{"id": 3, "firstname": "Meg", "age": 11, "country": "France", "profession": "unemployed", "category": "Books", "product_name": "Kids Story Book"}
INFO:root:{"id": 4, "firstname": "Rani", "age": 53, "country": "Spain", "profession": "doctor", "category": "Electronics", "product_name": "Smartphone Y"}

Let's add a transform - Filter. To use this transform you need to specify Filter condition.  Below you'll find an example with a condition written in Python. This pipeline will filter out records where the category is 'Electronics'. Verify this by scrolling to the bottom of the output logs

This `pipeline-filter-01.yaml` extends the previous pipeline by introducing a `Filter` transform. After reading the CSV data, the `Filter` transform will process each record, keeping only those where the `category` field is exactly 'Electronics'.

*   **`Filter`**: Configured with `language: python` and `keep: category == "Electronics"`, this transform evaluates the Python expression `category == "Electronics"` for each incoming record. Only records for which this expression evaluates to `True` will be passed on to the next transform.

After filtering, the `LogForTesting` transform will again print the remaining records. You should observe that only products belonging to the 'Electronics' category are logged.

In [7]:
%%writefile 'pipelines/pipeline-filter-01.yaml'
pipeline:
  type: chain
  transforms:
    - type: ReadFromCsv
      config:
        path: data/products.csv
    - type: Filter
      name: FilterWithCategory # Named to match input in Combine transform
      config:
        language: python
        keep: category == "Electronics"
    - type: LogForTesting

Writing pipelines/pipeline-filter-01.yaml


Executing `pipeline-filter-01.yaml` will run the pipeline with the added filtering logic. In the output logs, you should only see records where the `category` is 'Electronics'. For example, products like 'Laptop X', 'Smartphone Y', and 'Gaming Mouse' should appear, while 'The Great Novel' (category 'Books') should not. This demonstrates how to selectively process data based on specific conditions.

In [8]:
! python -m apache_beam.yaml.main --pipeline_spec_file=pipelines/pipeline-filter-01.yaml

INFO:root:Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
Building pipeline...
INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.
2026-04-08 06:19:38.887945: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 06:19:38.898420: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 06:19:38.931152: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775629178.985530    5310 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775629179.001370    5310 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attemptin

This `pipeline-custom-01.yaml` demonstrates a more complex data processing scenario, including filtering, grouping, and aggregation, finally writing the results to a new CSV file.

*   **`ReadFromCsv`**: Reads the initial `products.csv` file.
*   **`Filter`**: Similar to the previous example, it filters records to include only those where `category == "Electronics"`.
*   **`Combine`**: This is a powerful transform for aggregation. It groups the filtered records by `product_name` and then calculates the `num_sold` for each product. The `fn: count` applies a count aggregation to the `product_name` within each group, effectively counting how many times each electronic product appears.
*   **`WriteToCsv`**: The final step writes the aggregated results (product name and their counts) to a new CSV file named `electronics_filter.csv` in the current directory. The output CSV will contain two columns: `product_name` and `num_sold`.

In [9]:
%%writefile 'pipelines/pipeline-custom-01.yaml'
pipeline:
  type: chain
  transforms:
    - type: ReadFromCsv
      config:
        path: data/products.csv

    - type: Filter
      name: FilterWithCategory # Named to match input in Combine transform
      config:
        language: python
        keep: category == "Electronics"

    - type: Combine
      name: CountNumberSold
      config:
        group_by: product_name
        combine:
          num_sold:
            value: product_name
            fn: count

    - type: WriteToCsv
      config:
        path: electronics_filter.csv

Writing pipelines/pipeline-custom-01.yaml


In [ ]:
! python -m apache_beam.yaml.main --pipeline_spec_file=pipelines/pipeline-custom-01.yaml

INFO:root:Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
Building pipeline...
INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.
2026-04-08 06:21:04.217868: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 06:21:04.223449: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 06:21:04.238684: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775629264.264135    5752 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775629264.271783    5752 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attemptin

In [12]:
! head /content/electronics_filter.csv-00000-of-00001

product_name,num_sold
Gaming Mouse,2
Laptop Z,1
Laptop X,3
Smartphone Y,3
